# Example 2

In [ ]:
from mpi4py import MPI
from petsc4py import PETSc

import matplotlib.pyplot as plt
import numpy as np
import ufl
from dolfinx import fem, mesh
from dolfinx.fem.petsc import NonlinearProblem
from dolfinx.io import VTXWriter


def exact_solution(x, A):
    """MMS exact solution satisfying:
    - u = 0 at x = 0 (inlet)
    - du/dx = 0 at x = 1 (outlet, zero diffusive flux)
    - du/dy = 0 at y = 0 (symmetry / reflective)
    """
    return ufl.sin(ufl.pi * x[0] / 2) * (1 + A * ufl.cos(2 * ufl.pi * x[1]))


def solve_advection_diffusion(N, degree=1, D_value=0.01, w_max=1.0, A=0.5):
    msh = mesh.create_unit_square(
        MPI.COMM_WORLD,
        N,
        N,
        cell_type=mesh.CellType.triangle,
        diagonal=mesh.DiagonalType.crossed,
    )

    V = fem.functionspace(msh, ("DG", degree))

    u = fem.Function(V)
    v = ufl.TestFunction(V)

    x = ufl.SpatialCoordinate(msh)
    n = ufl.FacetNormal(msh)
    h = ufl.CellDiameter(msh)

    D = fem.Constant(msh, PETSc.ScalarType(D_value))

    # Velocity as an interpolated Function (exportable)
    import basix.ufl

    vel_el = basix.ufl.element(
        "Lagrange", msh.topology.cell_name(), 2, shape=(msh.geometry.dim,)
    )
    V_vel = fem.functionspace(msh, vel_el)
    w = fem.Function(V_vel, name="velocity")

    # Half-parabola Poiseuille-like velocity (x-direction only):
    #   w_x = w_max (1 - y^2)   -- zero at y=1 (wall), max at y=0 (symmetry)
    #   w_y = 0
    def velocity_func(x):
        values = np.zeros((2, x.shape[1]))
        values[0] = w_max * (1 - x[1] ** 2)
        return values

    w.interpolate(velocity_func)

    writer = VTXWriter(msh.comm, "velocity_field_MMS.bp", w, "BP5")
    writer.write(t=0)
    writer.close()

    u_exact = exact_solution(x, A)
    # Conservative source: div(w u) - div(D grad(u)) = f
    f = ufl.div(w * u_exact) - D * ufl.div(ufl.grad(u_exact))

    penalty = fem.Constant(msh, PETSc.ScalarType(100 * degree**2))

    # --- Mark boundary facets ---
    tdim = msh.topology.dim
    fdim = tdim - 1
    msh.topology.create_entities(fdim)

    inlet_id = 1  # x = 0
    outlet_id = 2  # x = 1
    wall_id = 3  # y = 1 (top, impermeable wall)
    symmetry_id = 4  # y = 0 (bottom, reflective / symmetry)

    inlet_facets = mesh.locate_entities_boundary(
        msh, fdim, lambda x: np.isclose(x[0], 0.0)
    )
    outlet_facets = mesh.locate_entities_boundary(
        msh, fdim, lambda x: np.isclose(x[0], 1.0)
    )
    wall_facets = mesh.locate_entities_boundary(
        msh, fdim, lambda x: np.isclose(x[1], 1.0)
    )
    symmetry_facets = mesh.locate_entities_boundary(
        msh, fdim, lambda x: np.isclose(x[1], 0.0)
    )

    facet_imap = msh.topology.index_map(fdim)
    num_facets = facet_imap.size_local + facet_imap.num_ghosts
    markers = np.zeros(num_facets, dtype=np.intc)
    markers[inlet_facets] = inlet_id
    markers[outlet_facets] = outlet_id
    markers[wall_facets] = wall_id
    markers[symmetry_facets] = symmetry_id

    indices = np.arange(num_facets, dtype=np.intc)
    mt = mesh.meshtags(msh, fdim, indices, markers)

    ds = ufl.Measure("ds", domain=msh, subdomain_data=mt)
    dS = ufl.dS
    dx = ufl.dx

    # Outflow indicator
    lmbda = ufl.conditional(ufl.gt(ufl.dot(w, n), 0), 1, 0)

    F = 0

    # ==================== ADVECTION (upwind DG) ====================

    # Volume (from integration by parts of div(w u))
    F += -ufl.inner(w * u, ufl.grad(v)) * dx

    # Interior faces (upwind numerical flux)
    F += ufl.inner(2 * ufl.avg(lmbda * w * u), ufl.jump(v, n)) * dS

    # --- Inlet (x=0): Dirichlet inflow, u = u_exact ---
    F += ufl.inner(lmbda * ufl.dot(w, n) * u, v) * ds(inlet_id)
    F += -ufl.inner((1 - lmbda) * ufl.dot(w, n) * u_exact, v) * ds(inlet_id)

    # --- Outlet (x=1): free outflow ---
    F += ufl.inner(lmbda * ufl.dot(w, n) * u, v) * ds(outlet_id)

    # --- Top wall (y=1): impermeable wall (zero total flux) ---
    #     w = 0 at y=1, so advective flux vanishes naturally.
    #     No advection terms needed.

    # --- Bottom symmetry (y=0): reflective (ghost = interior value) ---
    #     Numerical flux = (w·n) u_int  (no upwinding, mirror image)
    F += ufl.inner(ufl.dot(w, n) * u, v) * ds(symmetry_id)

    # ==================== DIFFUSION (SIPG) ====================

    # Volume
    F += D * ufl.inner(ufl.grad(u), ufl.grad(v)) * dx

    # Interior faces
    F += -D * ufl.inner(ufl.avg(ufl.grad(u)), ufl.jump(v, n)) * dS  # consistency
    F += -D * ufl.inner(ufl.jump(u, n), ufl.avg(ufl.grad(v))) * dS  # symmetry
    F += (
        D * (penalty / ufl.avg(h)) * ufl.inner(ufl.jump(u, n), ufl.jump(v, n)) * dS
    )  # penalty

    # --- Inlet (x=0): Nitsche for u = u_exact ---
    F += D * (
        -ufl.inner(ufl.grad(u), v * n) * ds(inlet_id)
        - ufl.inner(ufl.grad(v), (u - u_exact) * n) * ds(inlet_id)
        + (penalty / h) * ufl.inner(u - u_exact, v) * ds(inlet_id)
    )

    # --- Outlet (x=1): natural zero diffusive flux (∂u/∂n = 0) ---
    #     No Nitsche terms needed.

    # --- Top wall (y=1): natural zero diffusive flux (∂u/∂n = 0) ---
    #     No Nitsche terms needed (impermeable wall).

    # --- Bottom symmetry (y=0): natural zero diffusive flux (∂u/∂n = 0) ---
    #     No Nitsche terms needed.

    # ==================== SOURCE ====================
    F += -ufl.inner(f, v) * dx

    J = ufl.derivative(F, u)

    problem = NonlinearProblem(
        F,
        u,
        J=J,
        petsc_options_prefix="advecdiff",
        petsc_options={
            "snes_type": "newtonls",
            "snes_linesearch_type": "none",
            "snes_rtol": 1e-10,
            "snes_atol": 1e-10,
            "snes_max_it": 20,
            "ksp_type": "preonly",
            "pc_type": "lu",
        },
    )

    u = problem.solve()
    u.x.scatter_forward()

    writer = VTXWriter(msh.comm, "solution_MMS.bp", u, "BP5")
    writer.write(t=0)
    writer.close()

    # Compute L2 error
    error_form = fem.form((u - u_exact) ** 2 * dx)
    local_error = fem.assemble_scalar(error_form)
    l2_error = np.sqrt(msh.comm.allreduce(local_error, op=MPI.SUM))

    return l2_error


def convergence_test(degree=1):
    Ns = [8, 16, 32, 64, 128]
    errors = []

    for N in Ns:
        error = solve_advection_diffusion(N, degree=degree)
        errors.append(error)
        if MPI.COMM_WORLD.rank == 0:
            print(f"N = {N:3d}, L2 error = {error:.6e}")

    if MPI.COMM_WORLD.rank == 0:
        h = np.array([1.0 / N for N in Ns], dtype=float)
        errors = np.array(errors)

        # Compute convergence rates
        rates = np.log(errors[:-1] / errors[1:]) / np.log(h[:-1] / h[1:])
        for i, rate in enumerate(rates):
            print(f"  h = {h[i]:.4f} -> {h[i + 1]:.4f}: rate = {rate:.2f}")

        # Plot
        plt.figure()
        plt.loglog(h, errors, "o-", label=f"DG({degree})")

        ref_order = degree + 1
        plt.loglog(
            h,
            errors[0] * (h / h[0]) ** ref_order,
            "--k",
            label=f"Order {ref_order}",
        )

        plt.xlabel("Element size (h)")
        plt.ylabel("L2 error")
        plt.legend()
        plt.grid(True, which="both", ls="--", lw=0.5)

        ax = plt.gca()
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

        plt.tight_layout()
        plt.savefig("convergence.png", dpi=150)
        plt.show()


if __name__ == "__main__":
    convergence_test(degree=1)
